# Exercise 9 - Stochastic Gradient Descent and Learning Rate

Estimated time : ~30 minutes

The goal of this exercise is to experiment with the learning rate, learning rate decay schedules, and the optimization algorithm using the MNIST dataset.

There are many, many tutorials on the web showing how to use the MNIST dataset with TensorFlow and with Keras. For example, [Not another MNIST tutorial with TensorFlow](https://www.oreilly.com/learning/not-another-mnist-tutorial-with-tensorflow)

Recommended Hardware accelerator: **T4 GPU**

First run the code below to fetch the MNIST dataset and plot a few samples.

### MNIST Dataset

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils    import to_categorical
import os

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

(x_train, y_train), (x_test, y_test) = mnist.load_data()

n_train = len(x_train)

print()
print(f'Read {n_train} training images')

# Plot a few examples so we can visualize the dataset
image_size = 28
m = 10

fig = plt.figure(1, figsize=(10,1))
for i in range(m):
    a = fig.add_subplot(1,m,i+1)
    plt.imshow(x_train[i])
plt.show()

### Data Preparation
- Each pixel will be treated as an independent feature. So, the input layer for one picture will have `28*28 = 764` neurons
- the output layer will have 10 neurons, corresponding to the digit recognized. We'll use one-hot encoding

In [ ]:
x_train = x_train.reshape(-1, 784).astype(np.float32)
x_test  = x_test.reshape(-1, 784).astype(np.float32)
n_labels = 10
y_train = to_categorical(y_train, n_labels)
y_test  = to_categorical(y_test, n_labels)

Now run the code below to create and run the TensorFlow graph.

In [ ]:
import tensorflow
from tensorflow.keras.utils      import to_categorical
from tensorflow.keras.models     import Sequential
from tensorflow.keras.layers     import Dense, Input
from tensorflow.keras.optimizers import SGD
from packaging import version

num_hidden     = 128
minibatch_size = 100
n_epochs       = 10
steps_per_epoch = n_train//minibatch_size

def get_acc(logs):
    if version.parse(tensorflow.version.VERSION) < version.parse("2.0.0"):
        return logs['acc']
    else:
        return logs['accuracy']

def build_and_run_graph(learning_rate):
    tensorflow.keras.backend.clear_session()

    model = Sequential()
    model.add(Input(shape=(image_size*image_size,)))
    model.add(Dense( units=num_hidden, activation='relu'))
    model.add(Dense(units=n_labels, activation='softmax'))
    model.compile(loss='categorical_crossentropy', optimizer=SGD(learning_rate=learning_rate), metrics=['accuracy'])

    # Define a Keras callback
    class acc_history(tensorflow.keras.callbacks.Callback):
        def on_train_begin(self, logs={}):
            self.acc = []

        # Store the accuracy at the end of each gradient descent step
        def on_batch_end(self, batch, logs={}):
            self.acc.append(get_acc(logs))

        # Print the accuracy at the end of each epoch
        def on_epoch_end(self, epochs, logs={}):
            print(f'Step = { epochs*steps_per_epoch:6}, loss = {logs["loss"]:6.3f}, '
                  f'minibatch accuracy = {get_acc(logs)*100:4.1f}')

    logs = acc_history()

    model.fit(x_train, y_train, epochs=n_epochs, batch_size=minibatch_size, shuffle=True, verbose=0, callbacks=[logs])

    plt.figure(1, figsize=(10,10))
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.plot(logs.acc)
    plt.show()

Run the TensorFlow graph and experiment with the learning rate. Make sure you observe the effect of a very low learning rate (slow learning) and a very high learning rate (instability).

In [ ]:
build_and_run_graph(learning_rate = 0.1)

Modify the TensorFlow graph above to add a learning rate decay schedule. Experiment with the values of the initial learning rate, the interval, and the decay rate.

Finally, replace the gradient descent optimizer with the Adam optimizer, and compare the results.

In [ ]:
#

#### Solution 

If you want some help with the answer, you can look at our answer.  Copy and paste necessary sections of the code in a new cell to run the exercise.  Do not click on the cell below unless you want to see the answer we provide!

<details>
    <summary> See our answer </summary>

    from tensorflow.keras.utils      import to_categorical
    from tensorflow.keras.models     import Sequential
    from tensorflow.keras.layers     import Dense, Input
    from tensorflow.keras.optimizers import SGD
    from packaging import version

    num_hidden     = 128
    minibatch_size = 100
    n_epochs       = 10
    steps_per_epoch = n_train//minibatch_size

    def get_acc(logs):
        if version.parse(tensorflow.version.VERSION) < version.parse("2.0.0"):
            return logs['acc']
        else:
            return logs['accuracy']

    def build_and_run_graph(learning_rate, decay_rate):
        tensorflow.keras.backend.clear_session()

        model = Sequential()
        model.add(Input(shape=(image_size*image_size,)))
        model.add(Dense(units=num_hidden, activation='relu'))
        model.add(Dense(units=n_labels, activation='softmax'))
        model.compile(loss='categorical_crossentropy', optimizer=SGD(learning_rate=learning_rate), metrics=['accuracy'])

        # Define a learning rate decay scheduler
        def exp_decay(epoch):
           return learning_rate * decay_rate ** epoch

        lr_scheduler = tensorflow.keras.callbacks.LearningRateScheduler(exp_decay)

        # Define a Keras callback
        class acc_history(tensorflow.keras.callbacks.Callback):
            def on_train_begin(self, logs={}):
                self.acc = []

            # Store the accuracy at the end of each gradient descent step
            def on_batch_end(self, batch, logs={}):
                self.acc.append(get_acc(logs))

            # Print the accuracy and the learning rate at the end of each epoch
            def on_epoch_end(self, epochs, logs={}):
                print(f'Step = {epochs*steps_per_epoch:6}, '
                      f'loss = {logs["loss"]:6.3f}, '
                      f'minibatch accuracy = {get_acc(logs)*100:4.1f}, '
                      f'rate = {tensorflow.keras.backend.get_value(self.model.optimizer.learning_rate):5.3f}')

        logs = acc_history()

        model.fit(x_train, y_train, epochs=n_epochs, batch_size=minibatch_size, shuffle=True, verbose=0,
                  callbacks=[lr_scheduler, logs])

        plt.figure(1, figsize=(10,10))
        plt.xlabel('Epoch')
        plt.ylabel('Accuracy')
        plt.plot(logs.acc)
        plt.show()

        build_and_run_graph(learning_rate = 1.0, decay_rate = 0.85)
</details>